In [ ]:
#!/usr/bin/env python3
"""
Simplified Eye Disease Detection Training Script
Downloads Kaggle dataset and trains the model
"""

import os
import sys
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check if required packages are available
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report
    from kaggle.api.kaggle_api_extended import KaggleApi
except ImportError as e:
    print(f"Missing required package: {e}")
    print("Please run: pip install tensorflow opencv-python scikit-learn kaggle matplotlib tqdm")
    sys.exit(1)

print("✅ All required packages are available!")

class SimpleFundusTrainer:
    def __init__(self):
        self.dataset_path = "fundus_dataset"
        self.model_path = "models"
        self.image_size = (224, 224)
        self.batch_size = 32
        
        # Create directories
        Path(self.model_path).mkdir(exist_ok=True)
        Path("logs").mkdir(exist_ok=True)
    
    def download_dataset(self):
        """Download the Kaggle dataset"""
        print("📥 Downloading dataset from Kaggle...")
        
        try:
            api = KaggleApi()
            api.authenticate()
            
            # Download dataset
            api.dataset_download_files(
                "linchundan/fundusimage1000",
                path=self.dataset_path,
                unzip=True
            )
            print("✅ Dataset downloaded successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print("Please check your Kaggle API setup:")
            print("1. Ensure kaggle.json is in ~/.kaggle/")
            print("2. Run: chmod 600 ~/.kaggle/kaggle.json")
            return False
    
    def preprocess_image(self, image_path):
        """Simple but effective image preprocessing"""
        try:
            # Read image
            image = cv2.imread(str(image_path))
            if image is None:
                return None
            
            # Convert to RGB
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Resize
            image = cv2.resize(image, self.image_size)
            
            # Normalize
            image = image.astype(np.float32) / 255.0
            
            return image
        except Exception:
            return None
    
    def load_dataset(self):
        """Load and preprocess the dataset"""
        print("📁 Loading dataset...")
        
        images = []
        labels = []
        
        dataset_path = Path(self.dataset_path)
        categories = [d for d in dataset_path.iterdir() if d.is_dir()]
        
        print(f"Found {len(categories)} categories")
        
        for category_dir in tqdm(categories, desc="Processing categories"):
            category_name = category_dir.name
            
            # Get image files
            image_files = (
                list(category_dir.glob("*.jpg")) + 
                list(category_dir.glob("*.jpeg")) + 
                list(category_dir.glob("*.png"))
            )
            
            # Process images
            for image_file in tqdm(image_files, desc=f"Processing {category_name}", leave=False):
                processed_image = self.preprocess_image(image_file)
                if processed_image is not None:
                    images.append(processed_image)
                    labels.append(category_name)
        
        # Convert to arrays
        X = np.array(images)
        y = np.array(labels)
        
        # Encode labels
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
        
        print(f"✅ Dataset loaded: {len(X)} images, {len(label_encoder.classes_)} categories")
        
        return X, y_encoded, label_encoder.classes_
    
    def create_model(self, num_classes):
        """Create the neural network model"""
        print("🧠 Creating model...")
        
        # Use EfficientNet as backbone
        base_model = EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.image_size, 3)
        )
        base_model.trainable = False
        
        # Add custom layers
        model = keras.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation='softmax')
        ])
        
        # Compile
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        print(f"✅ Model created with {model.count_params():,} parameters")
        return model
    
    def train_model(self, model, X_train, y_train, X_val, y_val, epochs=25):
        """Train the model"""
        print("🚀 Starting training...")
        
        # Data augmentation
        datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        
        # Callbacks
        callbacks = [
            EarlyStopping(patience=8, restore_best_weights=True),
            ReduceLROnPlateau(patience=4, factor=0.5)
        ]
        
        # Train
        history = model.fit(
            datagen.flow(X_train, y_train, batch_size=self.batch_size),
            epochs=epochs,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def evaluate_model(self, model, X_test, y_test, class_names):
        """Evaluate the trained model"""
        print("📊 Evaluating model...")
        
        # Get predictions
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        y_pred = model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Classification report
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred_classes, target_names=class_names))
        
        return test_accuracy, test_loss
    
    def save_model(self, model, class_names, accuracy):
        """Save the trained model and metadata"""
        print("💾 Saving model...")
        
        # Save model
        model_file = f"{self.model_path}/eye_disease_model.h5"
        model.save(model_file)
        
        # Save class names
        classes_file = f"{self.model_path}/classes.json"
        with open(classes_file, 'w') as f:
            json.dump(class_names.tolist(), f, indent=2)
        
        # Save metadata
        metadata = {
            'model_file': model_file,
            'classes_file': classes_file,
            'num_classes': len(class_names),
            'image_size': self.image_size,
            'test_accuracy': float(accuracy),
            'class_names': class_names.tolist()
        }
        
        metadata_file = f"{self.model_path}/model_info.json"
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to: {model_file}")
        print(f"✅ Classes saved to: {classes_file}")
        print(f"✅ Metadata saved to: {metadata_file}")
    
    def run_complete_training(self):
        """Run the complete training pipeline"""
        print("🎯 Eye Disease Detection - Training Pipeline")
        print("=" * 50)
        
        # Step 1: Download dataset
        if not os.path.exists(self.dataset_path):
            if not self.download_dataset():
                return False
        else:
            print("📁 Dataset already exists, skipping download")
        
        # Step 2: Load dataset
        X, y, class_names = self.load_dataset()
        
        # Step 3: Split dataset
        print("✂️ Splitting dataset...")
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )
        
        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Step 4: Create and train model
        model = self.create_model(len(class_names))
        history = self.train_model(model, X_train, y_train, X_val, y_val)
        
        # Step 5: Evaluate model
        accuracy, loss = self.evaluate_model(model, X_test, y_test, class_names)
        
        # Step 6: Save model
        self.save_model(model, class_names, accuracy)
        
        print("\n🎉 Training completed successfully!")
        print(f"Final Test Accuracy: {accuracy:.4f}")
        print(f"Model saved in '{self.model_path}' directory")
        
        return True

def main():
    """Main function"""
    print("Eye Disease Detection Training")
    print("Using dataset: linchundan/fundusimage1000")
    print()
    
    # Check TensorFlow
    print(f"TensorFlow version: {tf.__version__}")
    print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
    print()
    
    # Run training
    trainer = SimpleFundusTrainer()
    success = trainer.run_complete_training()
    
    if success:
        print("\n✅ All done! Your model is ready for use.")
        print("\nNext steps:")
        print("1. Test your model with new images")
        print("2. Deploy using FastAPI for predictions")
        print("3. Create a web interface for doctors to use")
    else:
        print("\n❌ Training failed. Please check the errors above.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Fixed Eye Disease Detection Training Script
Downloads Kaggle dataset and trains the model with improved dataset loading
"""

import os
import sys
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check if required packages are available
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report
    from kaggle.api.kaggle_api_extended import KaggleApi
except ImportError as e:
    print(f"Missing required package: {e}")
    print("Please run: pip install tensorflow opencv-python scikit-learn kaggle matplotlib tqdm")
    sys.exit(1)

print("✅ All required packages are available!")

class SimpleFundusTrainer:
    def __init__(self):
        self.dataset_path = "fundus_dataset"
        self.model_path = "models"
        self.image_size = (224, 224)
        self.batch_size = 32
        
        # Create directories
        Path(self.model_path).mkdir(exist_ok=True)
        Path("logs").mkdir(exist_ok=True)
    
    def download_dataset(self):
        """Download the Kaggle dataset"""
        print("📥 Downloading dataset from Kaggle...")
        
        try:
            api = KaggleApi()
            api.authenticate()
            
            # Download dataset
            api.dataset_download_files(
                "linchundan/fundusimage1000",
                path=self.dataset_path,
                unzip=True
            )
            print("✅ Dataset downloaded successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print("Please check your Kaggle API setup:")
            print("1. Ensure kaggle.json is in ~/.kaggle/")
            print("2. Run: chmod 600 ~/.kaggle/kaggle.json")
            return False
    
    def explore_dataset_structure(self):
        """Explore and print the dataset structure"""
        print("🔍 Exploring dataset structure...")
        dataset_path = Path(self.dataset_path)
        
        if not dataset_path.exists():
            print(f"❌ Dataset path {dataset_path} does not exist!")
            return
        
        print(f"📁 Dataset root: {dataset_path}")
        
        # Walk through all directories and files
        for root, dirs, files in os.walk(dataset_path):
            level = root.replace(str(dataset_path), '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            
            # Show first few files in each directory
            subindent = ' ' * 2 * (level + 1)
            image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
            
            if image_files:
                print(f"{subindent}📸 {len(image_files)} image files")
                if len(image_files) <= 5:
                    for f in image_files:
                        print(f"{subindent}  - {f}")
                else:
                    for f in image_files[:3]:
                        print(f"{subindent}  - {f}")
                    print(f"{subindent}  ... and {len(image_files) - 3} more")
            
            other_files = [f for f in files if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
            if other_files:
                print(f"{subindent}📄 Other files: {other_files[:5]}")
    
    def preprocess_image(self, image_path):
        """Simple but effective image preprocessing"""
        try:
            # Read image
            image = cv2.imread(str(image_path))
            if image is None:
                return None
            
            # Convert to RGB
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Resize
            image = cv2.resize(image, self.image_size)
            
            # Normalize
            image = image.astype(np.float32) / 255.0
            
            return image
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
    
    def load_dataset(self):
        """Load and preprocess the dataset with improved directory traversal"""
        print("📁 Loading dataset...")
        
        images = []
        labels = []
        
        dataset_path = Path(self.dataset_path)
        
        # Check if there's a nested 1000images directory
        if (dataset_path / "1000images").exists():
            print("Found nested 1000images directory, using that as root")
            dataset_path = dataset_path / "1000images"
        
        # First, explore the structure
        self.explore_dataset_structure()
        
        # Try different strategies to find images
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
        
        # Strategy 1: Look for subdirectories as categories
        subdirs = [d for d in dataset_path.iterdir() if d.is_dir()]
        
        if subdirs:
            print(f"Found {len(subdirs)} subdirectories, treating as categories")
            for category_dir in tqdm(subdirs, desc="Processing categories"):
                category_name = category_dir.name
                print(f"Processing category: {category_name}")
                
                # Get image files recursively
                image_files = []
                for ext in image_extensions:
                    image_files.extend(category_dir.rglob(f"*{ext}"))
                    image_files.extend(category_dir.rglob(f"*{ext.upper()}"))
                
                print(f"Found {len(image_files)} images in {category_name}")
                
                # Process images
                processed_count = 0
                for image_file in tqdm(image_files, desc=f"Processing {category_name}", leave=False):
                    processed_image = self.preprocess_image(image_file)
                    if processed_image is not None:
                        images.append(processed_image)
                        labels.append(category_name)
                        processed_count += 1
                
                print(f"Successfully processed {processed_count}/{len(image_files)} images from {category_name}")
        
        else:
            # Strategy 2: All images in root directory - create single category or use filename patterns
            print("No subdirectories found, looking for images in root directory")
            
            image_files = []
            for ext in image_extensions:
                image_files.extend(dataset_path.glob(f"*{ext}"))
                image_files.extend(dataset_path.glob(f"*{ext.upper()}"))
            
            print(f"Found {len(image_files)} images in root directory")
            
            if image_files:
                # Try to extract categories from filenames
                for image_file in tqdm(image_files, desc="Processing images"):
                    processed_image = self.preprocess_image(image_file)
                    if processed_image is not None:
                        # Try to extract category from filename
                        filename = image_file.stem.lower()
                        
                        # Common patterns for medical image datasets
                        if any(word in filename for word in ['normal', 'healthy']):
                            category = 'normal'
                        elif any(word in filename for word in ['diabetic', 'dr', 'retinopathy']):
                            category = 'diabetic_retinopathy'
                        elif any(word in filename for word in ['glaucoma']):
                            category = 'glaucoma'
                        elif any(word in filename for word in ['cataract']):
                            category = 'cataract'
                        else:
                            # Default category
                            category = 'eye_disease'
                        
                        images.append(processed_image)
                        labels.append(category)
        
        if len(images) == 0:
            print("❌ No images were loaded! Please check the dataset structure.")
            return None, None, None
        
        # Convert to arrays
        X = np.array(images)
        y = np.array(labels)
        
        # Encode labels
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
        
        print(f"✅ Dataset loaded: {len(X)} images, {len(label_encoder.classes_)} categories")
        print(f"Categories: {list(label_encoder.classes_)}")
        
        # Print class distribution
        unique, counts = np.unique(y, return_counts=True)
        print("\nClass distribution:")
        for class_name, count in zip(unique, counts):
            print(f"  {class_name}: {count} images")
        
        return X, y_encoded, label_encoder.classes_
    
    def create_model(self, num_classes):
        """Create the neural network model"""
        print("🧠 Creating model...")
        
        # Use EfficientNet as backbone
        base_model = EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.image_size, 3)
        )
        base_model.trainable = False
        
        # Add custom layers
        model = keras.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation='softmax')
        ])
        
        # Compile
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        print(f"✅ Model created with {model.count_params():,} parameters")
        return model
    
    def train_model(self, model, X_train, y_train, X_val, y_val, epochs=25):
        """Train the model"""
        print("🚀 Starting training...")
        
        # Data augmentation
        datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1,
            brightness_range=[0.8, 1.2],
            fill_mode='nearest'
        )
        
        # Callbacks
        callbacks = [
            EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(patience=4, factor=0.5, verbose=1)
        ]
        
        # Train
        history = model.fit(
            datagen.flow(X_train, y_train, batch_size=self.batch_size),
            steps_per_epoch=len(X_train) // self.batch_size,
            epochs=epochs,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def evaluate_model(self, model, X_test, y_test, class_names):
        """Evaluate the trained model"""
        print("📊 Evaluating model...")
        
        # Get predictions
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        y_pred = model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Classification report
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred_classes, target_names=class_names))
        
        return test_accuracy, test_loss
    
    def save_model(self, model, class_names, accuracy):
        """Save the trained model and metadata"""
        print("💾 Saving model...")
        
        # Save model
        model_file = f"{self.model_path}/eye_disease_model.h5"
        model.save(model_file)
        
        # Save class names
        classes_file = f"{self.model_path}/classes.json"
        with open(classes_file, 'w') as f:
            json.dump(class_names.tolist(), f, indent=2)
        
        # Save metadata
        metadata = {
            'model_file': model_file,
            'classes_file': classes_file,
            'num_classes': len(class_names),
            'image_size': self.image_size,
            'test_accuracy': float(accuracy),
            'class_names': class_names.tolist()
        }
        
        metadata_file = f"{self.model_path}/model_info.json"
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Model saved to: {model_file}")
        print(f"✅ Classes saved to: {classes_file}")
        print(f"✅ Metadata saved to: {metadata_file}")
    
    def run_complete_training(self):
        """Run the complete training pipeline"""
        print("🎯 Eye Disease Detection - Training Pipeline")
        print("=" * 50)
        
        # Step 1: Download dataset
        if not os.path.exists(self.dataset_path):
            if not self.download_dataset():
                return False
        else:
            print("📁 Dataset already exists, skipping download")
        
        # Step 2: Load dataset
        result = self.load_dataset()
        if result[0] is None:
            print("❌ Failed to load dataset. Exiting.")
            return False
        
        X, y, class_names = result
        
        # Check if we have enough data
        if len(X) < 10:
            print(f"❌ Not enough images loaded ({len(X)}). Need at least 10 images to train.")
            return False
        
        # Step 3: Split dataset
        print("✂️ Splitting dataset...")
        
        # Adjust test_size based on dataset size
        if len(X) < 100:
            test_size = 0.2
            val_size = 0.2
        else:
            test_size = 0.3
            val_size = 0.5
        
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=val_size, random_state=42, stratify=y_temp
        )
        
        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Step 4: Create and train model
        model = self.create_model(len(class_names))
        history = self.train_model(model, X_train, y_train, X_val, y_val)
        
        # Step 5: Evaluate model
        accuracy, loss = self.evaluate_model(model, X_test, y_test, class_names)
        
        # Step 6: Save model
        self.save_model(model, class_names, accuracy)
        
        print("\n🎉 Training completed successfully!")
        print(f"Final Test Accuracy: {accuracy:.4f}")
        print(f"Model saved in '{self.model_path}' directory")
        
        return True

def main():
    """Main function"""
    print("Eye Disease Detection Training")
    print("Using dataset: linchundan/fundusimage1000")
    print()
    
    # Check TensorFlow
    print(f"TensorFlow version: {tf.__version__}")
    print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
    print()
    
    # Run training
    trainer = SimpleFundusTrainer()
    success = trainer.run_complete_training()
    
    if success:
        print("\n✅ All done! Your model is ready for use.")
        print("\nNext steps:")
        print("1. Test your model with new images")
        print("2. Deploy using FastAPI for predictions")
        print("3. Create a web interface for doctors to use")
    else:
        print("\n❌ Training failed. Please check the errors above.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Dataset Reorganization Script
Fixes the nested directory structure and class imbalance
"""

import os
import shutil
from pathlib import Path
from collections import Counter
import json

def reorganize_dataset():
    """Reorganize the dataset to fix the nested structure"""
    
    print("🔧 DATASET REORGANIZATION")
    print("=" * 40)
    
    source_dir = Path("fundus_dataset/1000images")
    target_dir = Path("fundus_dataset_fixed")
    
    if not source_dir.exists():
        print("❌ Source directory doesn't exist!")
        return False
    
    # Create target directory
    target_dir.mkdir(exist_ok=True)
    
    # Get all subdirectories (disease categories)
    categories = [d for d in source_dir.iterdir() if d.is_dir() and d.name != "1000images"]
    
    print(f"Found {len(categories)} disease categories")
    
    category_counts = {}
    total_images = 0
    
    # Process each category
    for category in categories:
        category_name = category.name
        
        # Skip the problematic nested folder
        if category_name == "1000images":
            continue
            
        print(f"Processing: {category_name}")
        
        # Create target category directory
        target_category = target_dir / category_name
        target_category.mkdir(exist_ok=True)
        
        # Find all images in this category
        image_files = []
        for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
            image_files.extend(category.glob(f"*{ext}"))
            image_files.extend(category.glob(f"*{ext.upper()}"))
        
        # Copy images
        copied_count = 0
        for img_file in image_files:
            try:
                target_file = target_category / img_file.name
                shutil.copy2(img_file, target_file)
                copied_count += 1
            except Exception as e:
                print(f"Error copying {img_file}: {e}")
        
        category_counts[category_name] = copied_count
        total_images += copied_count
        print(f"  Copied {copied_count} images")
    
    print(f"\n📊 REORGANIZATION SUMMARY:")
    print(f"Total images processed: {total_images}")
    print(f"Total categories: {len(category_counts)}")
    
    # Show category distribution
    print(f"\nCategory distribution:")
    for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {count} images")
    
    return True, category_counts

def filter_categories_by_size(min_images=10):
    """Filter categories to only include those with enough samples"""
    
    print(f"\n🔍 FILTERING CATEGORIES (minimum {min_images} images)")
    print("=" * 50)
    
    source_dir = Path("fundus_dataset_fixed")
    target_dir = Path("fundus_dataset_balanced")
    
    if not source_dir.exists():
        print("❌ Fixed dataset doesn't exist!")
        return False
    
    target_dir.mkdir(exist_ok=True)
    
    valid_categories = []
    
    for category_dir in source_dir.iterdir():
        if not category_dir.is_dir():
            continue
            
        # Count images
        image_count = len(list(category_dir.glob("*.jpg")) + 
                         list(category_dir.glob("*.jpeg")) + 
                         list(category_dir.glob("*.png")))
        
        if image_count >= min_images:
            print(f"✅ {category_dir.name}: {image_count} images - KEEPING")
            
            # Copy entire category
            target_category = target_dir / category_dir.name
            if target_category.exists():
                shutil.rmtree(target_category)
            shutil.copytree(category_dir, target_category)
            valid_categories.append((category_dir.name, image_count))
        else:
            print(f"❌ {category_dir.name}: {image_count} images - REMOVING (too few)")
    
    print(f"\n📊 FINAL DATASET:")
    print(f"Valid categories: {len(valid_categories)}")
    total_images = sum(count for _, count in valid_categories)
    print(f"Total images: {total_images}")
    
    return True, valid_categories

def create_simplified_categories():
    """Create simplified medical categories"""
    
    print(f"\n🏥 CREATING SIMPLIFIED MEDICAL CATEGORIES")
    print("=" * 45)
    
    source_dir = Path("fundus_dataset_fixed")
    target_dir = Path("fundus_dataset_medical")
    
    if not source_dir.exists():
        print("❌ Fixed dataset doesn't exist!")
        return False
    
    target_dir.mkdir(exist_ok=True)
    
    # Define medical category mappings
    medical_categories = {
        'normal': ['0.0.Normal', '0.1.Tessellated fundus'],
        'diabetic_retinopathy': ['0.3.DR1', '1.0.DR2', '1.1.DR3'],
        'glaucoma': ['10.0.Possible glaucoma', '10.1.Optic atrophy', '0.2.Large optic cup'],
        'vascular_disorders': ['2.0.BRVO', '2.1.CRVO', '11.Severe hypertensive retinopathy'],
        'macular_disorders': ['6.Maculopathy', '7.ERM', '8.MH', '5.0.CSCR'],
        'retinal_detachment': ['4.Rhegmatogenous RD'],
        'other_pathology': []  # Will collect remaining categories
    }
    
    # Create target directories
    for med_cat in medical_categories.keys():
        (target_dir / med_cat).mkdir(exist_ok=True)
    
    category_counts = {}
    processed_categories = set()
    
    # Map existing categories to medical categories
    for med_cat, orig_categories in medical_categories.items():
        count = 0
        for orig_cat in orig_categories:
            source_cat_dir = source_dir / orig_cat
            if source_cat_dir.exists():
                processed_categories.add(orig_cat)
                
                # Copy all images from original category
                target_cat_dir = target_dir / med_cat
                
                for img_file in source_cat_dir.glob("*"):
                    if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
                        new_name = f"{orig_cat}_{img_file.name}"
                        shutil.copy2(img_file, target_cat_dir / new_name)
                        count += 1
        
        category_counts[med_cat] = count
        print(f"{med_cat}: {count} images")
    
    # Handle remaining categories
    other_count = 0
    for category_dir in source_dir.iterdir():
        if category_dir.is_dir() and category_dir.name not in processed_categories:
            target_cat_dir = target_dir / 'other_pathology'
            
            for img_file in category_dir.glob("*"):
                if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
                    new_name = f"{category_dir.name}_{img_file.name}"
                    shutil.copy2(img_file, target_cat_dir / new_name)
                    other_count += 1
    
    category_counts['other_pathology'] = other_count
    print(f"other_pathology: {other_count} images")
    
    # Remove empty categories
    final_categories = {k: v for k, v in category_counts.items() if v > 0}
    
    for cat_name, count in final_categories.items():
        if count == 0:
            shutil.rmtree(target_dir / cat_name)
    
    print(f"\n📊 MEDICAL CATEGORIES SUMMARY:")
    for cat, count in final_categories.items():
        if count > 0:
            print(f"  {cat}: {count} images")
    
    return True, final_categories

def main():
    """Main reorganization process"""
    
    print("🏥 EYE DISEASE DATASET REORGANIZER")
    print("=" * 50)
    
    # Step 1: Fix nested structure
    print("\n1️⃣ FIXING NESTED DIRECTORY STRUCTURE")
    success, _ = reorganize_dataset()
    if not success:
        return
    
    # Step 2: Create simplified medical categories
    print("\n2️⃣ CREATING MEDICAL CATEGORIES")
    success, medical_cats = create_simplified_categories()
    if not success:
        return
    
    # Step 3: Update training script path
    print("\n3️⃣ NEXT STEPS:")
    print("✅ Dataset reorganized successfully!")
    print("✅ Update your training script to use: 'fundus_dataset_medical'")
    print("✅ Expected categories:", list(medical_cats.keys()))
    
    # Save configuration
    config = {
        'dataset_path': 'fundus_dataset_medical',
        'categories': medical_cats,
        'total_images': sum(medical_cats.values()),
        'num_classes': len([k for k, v in medical_cats.items() if v > 0])
    }
    
    with open('dataset_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"✅ Configuration saved to dataset_config.json")
    print(f"\n🎯 Ready for proper medical AI training!")

if __name__ == "__main__":
    main()

In [ ]:
#improved model

In [ ]:
#!/usr/bin/env python3
"""
Improved Eye Disease Detection Training Script
Addresses class imbalance, adds better data augmentation, and improves model architecture
"""

import os
import sys
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter

# Check if required packages are available
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    from sklearn.model_selection import train_test_split, StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report, confusion_matrix
    from sklearn.utils.class_weight import compute_class_weight
    from kaggle.api.kaggle_api_extended import KaggleApi
    import seaborn as sns
except ImportError as e:
    print(f"Missing required package: {e}")
    print("Please run: pip install tensorflow opencv-python scikit-learn kaggle matplotlib tqdm seaborn")
    sys.exit(1)

print("✅ All required packages are available!")

class ImprovedFundusTrainer:
    def __init__(self):
        self.dataset_path = "fundus_dataset_medical"
        self.model_path = "models"
        self.image_size = (224, 224)
        self.batch_size = 16  # Reduced for better gradient updates
        
        # Create directories
        Path(self.model_path).mkdir(exist_ok=True)
        Path("logs").mkdir(exist_ok=True)
        Path("plots").mkdir(exist_ok=True)
    
    def download_dataset(self):
        """Download the Kaggle dataset"""
        print("📥 Downloading dataset from Kaggle...")
        
        try:
            api = KaggleApi()
            api.authenticate()
            
            # Download dataset
            api.dataset_download_files(
                "linchundan/fundusimage1000",
                path=self.dataset_path,
                unzip=True
            )
            print("✅ Dataset downloaded successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print("Please check your Kaggle API setup:")
            print("1. Ensure kaggle.json is in ~/.kaggle/")
            print("2. Run: chmod 600 ~/.kaggle/kaggle.json")
            return False
    
    def enhanced_preprocess_image(self, image_path):
        """Enhanced image preprocessing with better techniques"""
        try:
            # Read image
            image = cv2.imread(str(image_path))
            if image is None:
                return None
            
            # Convert to RGB
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) for better contrast
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            lab[:,:,0] = clahe.apply(lab[:,:,0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
            
            # Resize with anti-aliasing
            image = cv2.resize(image, self.image_size, interpolation=cv2.INTER_LANCZOS4)
            
            # Normalize
            image = image.astype(np.float32) / 255.0
            
            return image
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
    
    def load_dataset_with_balancing(self):
        """Load dataset with improved balancing strategies"""
        print("📁 Loading dataset with class balancing...")
        
        images = []
        labels = []
        
        dataset_path = Path(self.dataset_path)
        
        # Check for nested directory
        if (dataset_path / "1000images").exists():
            dataset_path = dataset_path / "1000images"
        
        # Load images
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
        subdirs = [d for d in dataset_path.iterdir() if d.is_dir()]
        
        class_counts = {}
        
        for category_dir in tqdm(subdirs, desc="Loading categories"):
            category_name = category_dir.name
            
            # Get image files
            image_files = []
            for ext in image_extensions:
                image_files.extend(category_dir.rglob(f"*{ext}"))
                image_files.extend(category_dir.rglob(f"*{ext.upper()}"))
            
            class_images = []
            for image_file in image_files:
                processed_image = self.enhanced_preprocess_image(image_file)
                if processed_image is not None:
                    class_images.append(processed_image)
            
            if class_images:
                images.extend(class_images)
                labels.extend([category_name] * len(class_images))
                class_counts[category_name] = len(class_images)
        
        if len(images) == 0:
            return None, None, None
        
        # Print initial class distribution
        print("\nInitial class distribution:")
        for class_name, count in class_counts.items():
            print(f"  {class_name}: {count} images")
        
        # Apply data balancing strategies
        X, y = self.balance_dataset(images, labels, class_counts)
        
        # Encode labels
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
        
        print(f"✅ Balanced dataset: {len(X)} images, {len(label_encoder.classes_)} categories")
        
        return np.array(X), y_encoded, label_encoder.classes_
    
    def balance_dataset(self, images, labels, class_counts):
        """Apply multiple strategies to balance the dataset"""
        print("⚖️ Balancing dataset...")
        
        # Strategy 1: Oversample minority classes using augmentation
        target_samples = max(class_counts.values())  # Target the largest class size
        min_samples = 50  # Minimum samples per class after balancing
        
        # If target is too high, cap it
        if target_samples > 200:
            target_samples = 200
        
        balanced_images = []
        balanced_labels = []
        
        # Group images by class
        class_images = {}
        for img, label in zip(images, labels):
            if label not in class_images:
                class_images[label] = []
            class_images[label].append(img)
        
        # Balance each class
        for class_name, class_imgs in class_images.items():
            current_count = len(class_imgs)
            needed_samples = max(min_samples, min(target_samples, current_count * 3))
            
            print(f"Class {class_name}: {current_count} -> {needed_samples} samples")
            
            # Add original images
            balanced_images.extend(class_imgs)
            balanced_labels.extend([class_name] * len(class_imgs))
            
            # Generate additional samples if needed
            if needed_samples > current_count:
                additional_needed = needed_samples - current_count
                augmented_images = self.generate_augmented_samples(class_imgs, additional_needed)
                balanced_images.extend(augmented_images)
                balanced_labels.extend([class_name] * len(augmented_images))
        
        print(f"Final balanced dataset: {len(balanced_images)} images")
        return balanced_images, balanced_labels
    
    def generate_augmented_samples(self, images, count_needed):
        """Generate augmented samples for minority classes"""
        augmented = []
        
        for i in range(count_needed):
            # Randomly select an image to augment
            base_img = images[i % len(images)].copy()
            
            # Apply random augmentations
            augmented_img = self.random_augment(base_img)
            augmented.append(augmented_img)
        
        return augmented
    
    def random_augment(self, image):
        """Apply random augmentations to an image"""
        # Convert to uint8 for OpenCV operations
        img = (image * 255).astype(np.uint8)
        
        # Random rotation
        if np.random.random() < 0.7:
            angle = np.random.uniform(-20, 20)
            center = (img.shape[1]//2, img.shape[0]//2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
        
        # Random brightness/contrast
        if np.random.random() < 0.6:
            alpha = np.random.uniform(0.8, 1.3)  # Contrast
            beta = np.random.uniform(-20, 20)    # Brightness
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
        
        # Random horizontal flip
        if np.random.random() < 0.5:
            img = cv2.flip(img, 1)
        
        # Random noise
        if np.random.random() < 0.3:
            noise = np.random.normal(0, 5, img.shape).astype(np.uint8)
            img = cv2.add(img, noise)
        
        # Convert back to float32 and normalize
        return img.astype(np.float32) / 255.0
    
    def create_improved_model(self, num_classes):
        """Create an improved model architecture"""
        print("🧠 Creating improved model...")
        
        # Use EfficientNetB0 as backbone
        base_model = EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.image_size, 3)
        )
        
        # Unfreeze top layers for fine-tuning
        base_model.trainable = True
        for layer in base_model.layers[:-20]:
            layer.trainable = False
        
        # Create model with improved architecture
        inputs = keras.Input(shape=(*self.image_size, 3))
        
        # Data augmentation layer
        x = layers.RandomFlip("horizontal")(inputs)
        x = layers.RandomRotation(0.1)(x)
        x = layers.RandomZoom(0.1)(x)
        x = layers.RandomContrast(0.1)(x)
        
        # Base model
        x = base_model(x, training=False)
        
        # Custom head
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)
        
        x = layers.Dense(512, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        
        model = keras.Model(inputs, outputs)
        
        # Use different learning rates for base and head
        optimizer = keras.optimizers.Adam(learning_rate=1e-4)
        
        model.compile(
            optimizer=optimizer,
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', 'top_3_accuracy']
        )
        
        print(f"✅ Improved model created with {model.count_params():,} parameters")
        return model
    
    def train_with_class_weights(self, model, X_train, y_train, X_val, y_val, class_names, epochs=30):
        """Train model with class weights and improved callbacks"""
        print("🚀 Training with class weights...")
        
        # Calculate class weights
        class_weights = compute_class_weight(
            'balanced',
            classes=np.unique(y_train),
            y=y_train
        )
        class_weight_dict = dict(enumerate(class_weights))
        
        print("Class weights:")
        for i, (class_name, weight) in enumerate(zip(class_names, class_weights)):
            print(f"  {class_name}: {weight:.3f}")
        
        # Enhanced callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_accuracy',
                patience=12,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                patience=6,
                factor=0.3,
                min_lr=1e-7,
                verbose=1
            ),
            ModelCheckpoint(
                f"{self.model_path}/best_model.h5",
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            )
        ]
        
        # Advanced data augmentation
        datagen = ImageDataGenerator(
            rotation_range=20,
            width_shift_range=0.15,
            height_shift_range=0.15,
            horizontal_flip=True,
            zoom_range=0.15,
            brightness_range=[0.7, 1.3],
            shear_range=0.1,
            fill_mode='nearest'
        )
        
        # Train with class weights
        history = model.fit(
            datagen.flow(X_train, y_train, batch_size=self.batch_size),
            steps_per_epoch=len(X_train) // self.batch_size,
            epochs=epochs,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            class_weight=class_weight_dict,
            verbose=1
        )
        
        return history
    
    def evaluate_with_detailed_metrics(self, model, X_test, y_test, class_names):
        """Comprehensive model evaluation"""
        print("📊 Detailed model evaluation...")
        
        # Get predictions
        test_loss, test_accuracy, test_top3 = model.evaluate(X_test, y_test, verbose=0)
        y_pred = model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Top-3 Accuracy: {test_top3:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Classification report
        print("\nDetailed Classification Report:")
        report = classification_report(
            y_test, y_pred_classes, 
            target_names=class_names, 
            output_dict=True,
            zero_division=0
        )
        print(classification_report(y_test, y_pred_classes, target_names=class_names, zero_division=0))
        
        # Confusion matrix
        self.plot_confusion_matrix(y_test, y_pred_classes, class_names)
        
        return test_accuracy, test_loss, report
    
    def plot_confusion_matrix(self, y_true, y_pred, class_names):
        """Plot and save confusion matrix"""
        cm = confusion_matrix(y_true, y_pred)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(
            cm, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names
        )
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig('plots/confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✅ Confusion matrix saved to plots/confusion_matrix.png")
    
    def plot_training_history(self, history):
        """Plot training history"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Accuracy
        axes[0,0].plot(history.history['accuracy'], label='Training Accuracy')
        axes[0,0].plot(history.history['val_accuracy'], label='Validation Accuracy')
        axes[0,0].set_title('Model Accuracy')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Accuracy')
        axes[0,0].legend()
        axes[0,0].grid(True)
        
        # Loss
        axes[0,1].plot(history.history['loss'], label='Training Loss')
        axes[0,1].plot(history.history['val_loss'], label='Validation Loss')
        axes[0,1].set_title('Model Loss')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)
        
        # Top-3 Accuracy
        if 'top_3_accuracy' in history.history:
            axes[1,0].plot(history.history['top_3_accuracy'], label='Training Top-3 Accuracy')
            axes[1,0].plot(history.history['val_top_3_accuracy'], label='Validation Top-3 Accuracy')
            axes[1,0].set_title('Top-3 Accuracy')
            axes[1,0].set_xlabel('Epoch')
            axes[1,0].set_ylabel('Top-3 Accuracy')
            axes[1,0].legend()
            axes[1,0].grid(True)
        
        # Learning Rate (if available)
        if 'lr' in history.history:
            axes[1,1].plot(history.history['lr'])
            axes[1,1].set_title('Learning Rate')
            axes[1,1].set_xlabel('Epoch')
            axes[1,1].set_ylabel('Learning Rate')
            axes[1,1].set_yscale('log')
            axes[1,1].grid(True)
        
        plt.tight_layout()
        plt.savefig('plots/training_history.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✅ Training history saved to plots/training_history.png")
    
    def save_improved_model(self, model, class_names, accuracy, report):
        """Save model with comprehensive metadata"""
        print("💾 Saving improved model...")
        
        # Save model in Keras format
        model_file = f"{self.model_path}/eye_disease_model_improved.keras"
        model.save(model_file)
        
        # Save detailed metadata
        metadata = {
            'model_file': model_file,
            'num_classes': len(class_names),
            'image_size': self.image_size,
            'test_accuracy': float(accuracy),
            'class_names': class_names.tolist(),
            'classification_report': report,
            'model_architecture': 'EfficientNetB0 + Custom Head',
            'training_features': [
                'Class balancing with oversampling',
                'Advanced data augmentation',
                'Class weights',
                'Fine-tuning',
                'Enhanced preprocessing with CLAHE'
            ]
        }
        
        metadata_file = f"{self.model_path}/model_info_improved.json"
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        # Save class names
        classes_file = f"{self.model_path}/classes_improved.json"
        with open(classes_file, 'w') as f:
            json.dump(class_names.tolist(), f, indent=2)
        
        print(f"✅ Improved model saved to: {model_file}")
        print(f"✅ Metadata saved to: {metadata_file}")
        print(f"✅ Classes saved to: {classes_file}")
    
    def run_improved_training(self):
        """Run the complete improved training pipeline"""
        print("🎯 Improved Eye Disease Detection - Training Pipeline")
        print("=" * 60)
        
        # Step 1: Download dataset if needed
        if not os.path.exists(self.dataset_path):
            if not self.download_dataset():
                return False
        
        # Step 2: Load and balance dataset
        result = self.load_dataset_with_balancing()
        if result[0] is None:
            print("❌ Failed to load dataset. Exiting.")
            return False
        
        X, y, class_names = result
        
        if len(X) < 50:
            print(f"❌ Not enough images ({len(X)}). Need at least 50 for reliable training.")
            return False
        
        # Step 3: Stratified split
        print("✂️ Creating stratified splits...")
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )
        
        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Step 4: Create and train improved model
        model = self.create_improved_model(len(class_names))
        history = self.train_with_class_weights(
            model, X_train, y_train, X_val, y_val, class_names
        )
        
        # Step 5: Comprehensive evaluation
        accuracy, loss, report = self.evaluate_with_detailed_metrics(
            model, X_test, y_test, class_names
        )
        
        # Step 6: Save results
        self.save_improved_model(model, class_names, accuracy, report)
        self.plot_training_history(history)
        
        print("\n🎉 Improved training completed!")
        print(f"Final Test Accuracy: {accuracy:.4f}")
        print(f"Model and plots saved in respective directories")
        
        # Performance recommendations
        self.provide_performance_recommendations(accuracy, report)
        
        return True
    
    def provide_performance_recommendations(self, accuracy, report):
        """Provide recommendations based on performance"""
        print("\n📋 Performance Analysis & Recommendations:")
        print("-" * 50)
        
        if accuracy < 0.6:
            print("⚠️  Low accuracy detected. Consider:")
            print("   • Collecting more high-quality data")
            print("   • Trying different architectures (ResNet, DenseNet)")
            print("   • Adjusting hyperparameters")
        elif accuracy < 0.8:
            print("📈 Moderate performance. Improvements possible:")
            print("   • Fine-tune hyperparameters")
            print("   • Try ensemble methods")
            print("   • Experiment with different augmentation strategies")
        else:
            print("🎯 Good performance! Consider:")
            print("   • Cross-validation for robustness")
            print("   • Deployment optimization")
            print("   • Clinical validation")
        
        # Check for class-specific issues
        worst_classes = []
        for class_name in report:
            if isinstance(report[class_name], dict) and 'f1-score' in report[class_name]:
                if report[class_name]['f1-score'] < 0.5:
                    worst_classes.append(class_name)
        
        if worst_classes:
            print(f"\n🔍 Classes needing attention: {', '.join(worst_classes)}")
            print("   • Collect more samples for these classes")
            print("   • Review data quality for these categories")

def main():
    """Main function with improved error handling"""
    print("Improved Eye Disease Detection Training")
    print("Using enhanced techniques for better performance")
    print()
    
    # System info
    print(f"TensorFlow version: {tf.__version__}")
    print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
    
    # Set memory growth for GPU
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("✅ GPU memory growth enabled")
        except RuntimeError as e:
            print(f"GPU setup error: {e}")
    
    print()
    
    # Run improved training
    trainer = ImprovedFundusTrainer()
    success = trainer.run_improved_training()
    
    if success:
        print("\n✅ Training completed successfully!")
        print("\nFiles created:")
        print("• models/eye_disease_model_improved.keras - Main model")
        print("• models/model_info_improved.json - Detailed metadata")
        print("• plots/confusion_matrix.png - Performance visualization")
        print("• plots/training_history.png - Training progress")
        print("\nNext steps:")
        print("1. Test model with validation images")
        print("2. Consider clinical validation")
        print("3. Deploy for production use")
    else:
        print("\n❌ Training failed. Check errors above.")

if __name__ == "__main__":
    main()